In [1]:
import numpy as np
import pandas as pd

### Part I: Aggregate sales data
Load all transaction data, creating close year and month labels to support downstream model training and hyperparameter tuning

In [ ]:
months_2024 = [f"{m:02d}" for m in range(1, 13)]
files_2024 = [f"raw-data/CRMLSSold2024{m}.csv" for m in months_2024]
sold_2024 = pd.concat(pd.read_csv(f) for f in files_2024)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_60845/840400536.py:3: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_2024 = pd.concat(pd.read_csv(f) for f in files_2024)


In [ ]:
months_2025 = [f"{m:02d}" for m in range(1, 13)]
files_2025 = [f"raw-data/CRMLSSold2025{m}.csv" for m in months_2025]
sold_2025 = pd.concat(pd.read_csv(f) for f in files_2025)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_60845/4110478849.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_2025 = pd.concat(pd.read_csv(f) for f in files_2025)


In [ ]:
months_2026 = [f"{m:02d}" for m in range(1, 6)]
files_2026 = [f"raw-data/CRMLSSold2026{m}.csv" for m in months_2026]
sold_2026 = pd.concat(pd.read_csv(f) for f in files_2026)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_60845/4004451310.py:3: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_2026 = pd.concat(pd.read_csv(f) for f in files_2026)


In [39]:
sold = pd.concat([sold_2024, sold_2025, sold_2026])
sold.shape

(639254, 86)

### Part II: Filter for residential sales

Restrict analysis to single-family residential properties

In [53]:
sold_residential = sold[sold['PropertyType'] == 'Residential']
sold_sfr = sold_residential[sold_residential['PropertySubType'] == 'SingleFamilyResidence']
sold_sfr.shape

(321846, 86)

### Part III: Handle missing values

Profile missing values in key columns (e.g., close price, living area, bedrooms, bathrooms, lot size)

In [54]:
sold_sfr = sold_sfr.replace(r'^\s*$', np.nan, regex=True)
null_counts = sold_sfr.isna().sum()
null_percs =  sold_sfr.isna().mean() * 100
null_table = pd.DataFrame({
    'null count': null_counts,
    'null percentage': null_percs
})

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_60845/2925654690.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sold_sfr = sold_sfr.replace(r'^\s*$', np.nan, regex=True)


In [55]:
null_table.loc[[
    'ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt',
    'LotSizeArea', 'LotSizeAcres', 'LotSizeSquareFeet',
    'City', 'CountyOrParish', 'PostalCode', 'Latitude', 'Longitude',
    'CloseDate', 'ListingContractDate', 'PurchaseContractDate'
]]

,null count,null percentage
ClosePrice,2,0.000621
LivingArea,173,0.053752
BedroomsTotal,0,0.000000
BathroomsTotalInteger,50,0.015535
DaysOnMarket,0,0.000000
YearBuilt,232,0.072084
LotSizeArea,5516,1.713863
LotSizeAcres,5568,1.730020
LotSizeSquareFeet,5540,1.721320
City,246,0.076434


Drop records with missing close price, impute missing living area, bathrooms, and lot size with respective county medians, and flag records with missing city, postal code, latitude, or longitude

In [ ]:
# drop records with missing close price (target variable)
sold_sfr = sold_sfr.dropna(subset='ClosePrice')

In [ ]:
# impute records with missing configuration details
cols = ['LivingArea', 'BathroomsTotalInteger', 'LotSizeArea', 'LotSizeAcres', 'LotSizeSquareFeet']
sold_sfr[cols] = sold_sfr[cols].apply(pd.to_numeric)

for col in cols:
    sold_sfr[col] = sold_sfr[col].fillna(
        sold_sfr.groupby('CountyOrParish')[col].transform('median')
    )

In [ ]:
# flag records with missing location details
sold_sfr['missing_city'] = sold_sfr['City'].isna()
sold_sfr['missing_postalcode'] = sold_sfr['PostalCode'].isna()
sold_sfr['missing_lat'] = sold_sfr['Latitude'].isna()
sold_sfr['missing_lon'] = sold_sfr['Longitude'].isna()

### Part IV: Encode categorical columns

Use ordinal encoding to convert high-cardinality location columns to numeric

In [65]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.1 MB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 27.4 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 49.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [66]:
from sklearn.preprocessing import OrdinalEncoder

In [68]:
location_cols = ['City', 'CountyOrParish', 'PostalCode']

encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

sold_sfr[location_cols] = encoder.fit_transform(sold_sfr[location_cols].astype(str))

In [72]:
mapping_dict = {}

for col, categories in zip(location_cols, encoder.categories_):
    mapping_dict[col] = {
        category: code for code, category in enumerate(categories)
    }

# mapping_dict['City']
# mapping_dict['PostalCode']
mapping_dict['CountyOrParish']

{'Alameda': 0,
 'Alpine': 1,
 'Amador': 2,
 'Butte': 3,
 'Calaveras': 4,
 'Clark': 5,
 'Colusa': 6,
 'Contra Costa': 7,
 'Del Norte': 8,
 'El Dorado': 9,
 'Foreign Country': 10,
 'Fresno': 11,
 'Glenn': 12,
 'Humboldt': 13,
 'Imperial': 14,
 'Inyo': 15,
 'Kern': 16,
 'Kings': 17,
 'Lake': 18,
 'Lassen': 19,
 'Los Angeles': 20,
 'Madera': 21,
 'Marin': 22,
 'Mariposa': 23,
 'Mendocino': 24,
 'Merced': 25,
 'Modoc': 26,
 'Mono': 27,
 'Monterey': 28,
 'Napa': 29,
 'Nevada': 30,
 'Orange': 31,
 'Other': 32,
 'Other County': 33,
 'Other State': 34,
 'Placer': 35,
 'Plumas': 36,
 'Riverside': 37,
 'Sacramento': 38,
 'San Benito': 39,
 'San Bernardino': 40,
 'San Diego': 41,
 'San Francisco': 42,
 'San Joaquin': 43,
 'San Luis Obispo': 44,
 'San Mateo': 45,
 'Santa Barbara': 46,
 'Santa Clara': 47,
 'Santa Cruz': 48,
 'Shasta': 49,
 'Sierra': 50,
 'Siskiyou': 51,
 'Solano': 52,
 'Sonoma': 53,
 'Stanislaus': 54,
 'Sutter': 55,
 'Tehama': 56,
 'Trinity': 57,
 'Tulare': 58,
 'Tuolumne': 59,
 'Ve

### Part V: Create train-test split

Use records from the most recent month for testing and records in the preceding months for training

In [105]:
# hyperparameter to adjust
train_months = 12

close_month = pd.to_datetime(sold_sfr['CloseDate']).dt.to_period('M')
latest_month = close_month.max()

train_start = latest_month - train_months
train_set = sold_sfr[(close_month >= train_start) & (close_month < latest_month)]
test_set = sold_sfr[close_month == latest_month]

In [ ]:
train_set.to_csv("clean-data/sold_residential_metrics_outliers_removed.csv", index=False)

1        2026-05-28
3        2026-05-29
4        2026-05-28
5        2026-05-15
6        2026-05-29
            ...    
23226    2026-05-07
23228    2026-05-08
23236    2026-05-06
23238    2026-05-15
23256    2026-05-21
Name: CloseDate, Length: 12024, dtype: object